# P2 - Transfer Learning untuk Klasifikasi Tanaman UAV

Notebook ini mengadaptasi materi ResNet18 untuk dataset InterDuPa-UAV: Durian dan Papaya. Tiga strategi dilatih dengan split yang sama: feature extraction, partial fine-tuning, dan scratch. Jalankan seluruh sel secara berurutan di Kaggle atau lokal.

## 1. Configuration

Pada Kaggle, atur `DATASET_ROOT` ke folder input yang berisi dua folder kelas dan metadata. Untuk split bebas leakage, sediakan CSV `GROUP_MAPPING_PATH` dengan kolom `relative_path` dan `group_id`; setiap crop dari gambar UAV yang sama harus memiliki `group_id` yang sama.

In [ ]:
from pathlib import Path
import copy
import json
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from PIL import Image
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             precision_recall_fscore_support)
from sklearn.model_selection import StratifiedGroupKFold, StratifiedShuffleSplit
from torch import nn, optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

SEED = 42
BATCH_SIZE = 32
EPOCHS = 10
VAL_FRACTION = 0.20
NUM_WORKERS = 2

KAGGLE_DATASET_ROOT = Path('/kaggle/input/datasets/sangplanetdestroyer/dataset-p2-vision-dan-deep/datasets')
DATASET_ROOT = KAGGLE_DATASET_ROOT if KAGGLE_DATASET_ROOT.exists() else Path('datasets')
METADATA_PATH = DATASET_ROOT / 'Dataset-metadata.xlsx'
# Optional CSV with relative_path,group_id. Leave None only when no source-level mapping exists.
GROUP_MAPPING_PATH = None
GROUP_PATH_COLUMN = 'relative_path'
GROUP_ID_COLUMN = 'group_id'
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
RESULTS_DIR = OUTPUT_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Dataset root: {DATASET_ROOT.resolve()}')

In [ ]:
## 2. Dataset and metadata inspection
CLASS_DIRS = {
    'Durian': 'Durian (durio zibethinus)',
    'Papaya': 'Papaya (carica papaya)',
}
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

if not DATASET_ROOT.exists():
    raise FileNotFoundError(f'Dataset root not found: {DATASET_ROOT}')

records = []
for label, folder_name in CLASS_DIRS.items():
    class_dir = DATASET_ROOT / folder_name
    if not class_dir.exists():
        raise FileNotFoundError(f'Missing class directory: {class_dir}')
    records.extend({'path': path, 'label_name': label} for path in class_dir.rglob('*')
                   if path.suffix.lower() in IMAGE_EXTENSIONS)

df = pd.DataFrame(records).sort_values('path').reset_index(drop=True)
class_to_idx = {name: index for index, name in enumerate(CLASS_DIRS)}
df['label'] = df['label_name'].map(class_to_idx)
print(df.groupby('label_name').size().rename('images'))
print(f'Total images: {len(df)}')

if METADATA_PATH.exists():
    metadata = pd.read_excel(METADATA_PATH)
    display(metadata.head(10))
    print('Metadata columns:', metadata.columns.tolist())
else:
    print(f'Warning: metadata file not found at {METADATA_PATH}')

In [ ]:
## 3. Leakage-aware train/validation split
def make_split(frame):
    y = frame['label'].to_numpy()
    if GROUP_MAPPING_PATH is not None:
        mapping_path = Path(GROUP_MAPPING_PATH)
        if not mapping_path.exists():
            raise FileNotFoundError(f'Group mapping not found: {mapping_path}')
        mapping = pd.read_csv(mapping_path)
        required = {GROUP_PATH_COLUMN, GROUP_ID_COLUMN}
        if not required.issubset(mapping.columns):
            raise ValueError(f'Group mapping must contain {required}; found {mapping.columns.tolist()}')
        work = frame.copy()
        work['relative_path'] = work['path'].map(lambda value: str(value.relative_to(DATASET_ROOT)))
        work = work.merge(mapping[[GROUP_PATH_COLUMN, GROUP_ID_COLUMN]], left_on='relative_path',
                          right_on=GROUP_PATH_COLUMN, how='left', validate='one_to_one')
        if work[GROUP_ID_COLUMN].isna().any():
            missing = work.loc[work[GROUP_ID_COLUMN].isna(), 'relative_path'].head().tolist()
            raise ValueError(f'Unmapped images in group mapping, for example: {missing}')
        groups = work[GROUP_ID_COLUMN].astype(str).to_numpy()
        splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
        train_idx, val_idx = next(splitter.split(work, y, groups))
        assert set(groups[train_idx]).isdisjoint(set(groups[val_idx])), 'Group leakage detected'
        print(f'Group split: {len(set(groups))} source groups, with no group overlap.')
        return train_idx, val_idx

    print('WARNING: no per-image source group mapping was supplied. Using a stratified image split.')
    print('This is reproducible, but cannot rule out UAV-source leakage. Add GROUP_MAPPING_PATH before final reporting.')
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=SEED)
    return next(splitter.split(frame, y))

train_idx, val_idx = make_split(df)
assert not set(train_idx).intersection(val_idx)
train_df, val_df = df.iloc[train_idx].reset_index(drop=True), df.iloc[val_idx].reset_index(drop=True)
split_summary = pd.concat([train_df.assign(split='train'), val_df.assign(split='validation')])
display(pd.crosstab(split_summary['split'], split_summary['label_name'], normalize='index'))
print(f'Train: {len(train_df)} | Validation: {len(val_df)}')

In [ ]:
## 4. Augmentation, preprocessing, and dataloaders
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
valid_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class UAVPlantDataset(Dataset):
    def __init__(self, frame, transform):
        self.paths = frame['path'].tolist()
        self.labels = frame['label'].tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            image = image.convert('RGB')
        return self.transform(image), self.labels[index]

loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=device.type == 'cuda')
train_loader = DataLoader(UAVPlantDataset(train_df, train_transform), shuffle=True, **loader_kwargs)
val_loader = DataLoader(UAVPlantDataset(val_df, valid_transform), shuffle=False, **loader_kwargs)
class_names = list(CLASS_DIRS)
print(class_names)

In [ ]:
## 5. ResNet18 and training utilities
def build_model(strategy):
    if strategy == 'scratch':
        model = models.resnet18(weights=None)
    else:
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, len(class_names))

    if strategy == 'feature_extraction':
        for parameter in model.parameters():
            parameter.requires_grad = False
        for parameter in model.fc.parameters():
            parameter.requires_grad = True
        optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)
    elif strategy == 'partial_fine_tuning':
        for parameter in model.parameters():
            parameter.requires_grad = False
        for module in (model.layer4, model.fc):
            for parameter in module.parameters():
                parameter.requires_grad = True
        optimizer = optim.Adam([
            {'params': model.layer4.parameters(), 'lr': 1e-4},
            {'params': model.fc.parameters(), 'lr': 1e-3},
        ])
    elif strategy == 'scratch':
        optimizer = optim.Adam(model.parameters(), lr=1e-3)
    else:
        raise ValueError(f'Unknown strategy: {strategy}')
    return model.to(device), optimizer

def run_epoch(model, dataloader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    losses, predictions, targets = [], [], []
    for images, labels in tqdm(dataloader, leave=False):
        images, labels = images.to(device), labels.to(device)
        with torch.set_grad_enabled(training):
            outputs = model(images)
            loss = criterion(outputs, labels)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        losses.append(loss.item() * labels.size(0))
        predictions.extend(outputs.argmax(1).detach().cpu().tolist())
        targets.extend(labels.detach().cpu().tolist())
    return sum(losses) / len(dataloader.dataset), accuracy_score(targets, predictions), targets, predictions

def train_experiment(name, strategy):
    seed_everything()
    model, optimizer = build_model(strategy)
    criterion = nn.CrossEntropyLoss()
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
    history, best_state, best_acc, best_epoch = [], None, -np.inf, 0
    started = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc, _, _ = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc, _, _ = run_epoch(model, val_loader, criterion)
        scheduler.step()
        history.append({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc,
                        'val_loss': val_loss, 'val_acc': val_acc})
        print(f'{name} | {epoch:02d}/{EPOCHS} | train acc {train_acc:.4f} | val acc {val_acc:.4f}')
        if val_acc > best_acc:
            best_acc, best_epoch, best_state = val_acc, epoch, copy.deepcopy(model.state_dict())
    elapsed = time.perf_counter() - started
    model.load_state_dict(best_state)
    return {'name': name, 'strategy': strategy, 'model': model, 'history': pd.DataFrame(history),
            'best_val_accuracy': best_acc, 'best_epoch': best_epoch, 'training_time_seconds': elapsed}


In [ ]:
## 6. Run the three experiments
EXPERIMENTS = [
    ('Feature Extraction', 'feature_extraction'),
    ('Partial Fine-Tuning', 'partial_fine_tuning'),
    ('Scratch', 'scratch'),
]

experiments = [train_experiment(name, strategy) for name, strategy in EXPERIMENTS]
comparison = pd.DataFrame([{key: value for key, value in result.items()
                            if key not in {'model', 'history'}} for result in experiments])
comparison = comparison.sort_values('best_val_accuracy', ascending=False).reset_index(drop=True)
comparison.to_csv(RESULTS_DIR / 'comparison.csv', index=False)
display(comparison)

In [ ]:
## 7. Accuracy curves and best-model evaluation
plt.figure(figsize=(9, 5))
for result in experiments:
    history = result['history']
    plt.plot(history['epoch'], history['val_acc'], marker='o', label=result['name'])
plt.xlabel('Epoch')
plt.ylabel('Validation accuracy')
plt.title('Validation Accuracy: ResNet18 Strategies')
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'accuracy_curve.png', dpi=160)
plt.show()

best_result = next(result for result in experiments if result['name'] == comparison.iloc[0]['name'])
criterion = nn.CrossEntropyLoss()
val_loss, val_acc, y_true, y_pred = run_epoch(best_result['model'], val_loader, criterion)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
print(f"Best model: {best_result['name']}")
print(f'Validation loss: {val_loss:.4f} | accuracy: {val_acc:.4f} | F1: {f1:.4f}')
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.title(f'Confusion Matrix - {best_result["name"]}')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confusion_matrix.png', dpi=160)
plt.show()

In [ ]:
## 8. Inference latency and reproducibility artifacts
def measure_latency(model, dataloader, warmup=10, runs=100):
    model.eval()
    images, _ = next(iter(dataloader))
    images = images[:1].to(device)
    with torch.inference_mode():
        for _ in range(warmup):
            _ = model(images)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        started = time.perf_counter()
        for _ in range(runs):
            _ = model(images)
        if device.type == 'cuda':
            torch.cuda.synchronize()
    return (time.perf_counter() - started) * 1000 / runs

latency_ms = measure_latency(best_result['model'], val_loader)
comparison.loc[comparison['name'] == best_result['name'], 'inference_latency_ms_per_image'] = latency_ms
comparison.to_csv(RESULTS_DIR / 'comparison.csv', index=False)

config = {
    'seed': SEED, 'batch_size': BATCH_SIZE, 'epochs': EPOCHS, 'validation_fraction': VAL_FRACTION,
    'dataset_root': str(DATASET_ROOT), 'group_mapping_path': str(GROUP_MAPPING_PATH),
    'class_to_idx': class_to_idx, 'device': str(device),
}
with open(RESULTS_DIR / 'config.json', 'w') as file:
    json.dump(config, file, indent=2)

torch.save(best_result['model'].state_dict(), RESULTS_DIR / 'best_resnet18.pt')
print(f'Inference latency: {latency_ms:.2f} ms/image')
print(f'Artifacts saved in: {RESULTS_DIR.resolve()}')
display(comparison)

## 9. Final analysis

Gunakan `comparison.csv`, grafik akurasi, confusion matrix, dan latency untuk membandingkan tiga strategi. Laporkan apakah split memakai group mapping atau stratified image split; hasil dari split tanpa group mapping harus diberi keterbatasan potensi data leakage.

In [ ]:
## Dataset visualization
fig, axes = plt.subplots(1, len(class_names), figsize=(8, 4))
for axis, class_name in zip(np.atleast_1d(axes), class_names):
    sample_path = train_df.loc[train_df['label_name'] == class_name, 'path'].iloc[0]
    with Image.open(sample_path) as image:
        axis.imshow(image.convert('RGB'))
    axis.set_title(class_name)
    axis.axis('off')
plt.tight_layout()
plt.show()